In [ ]:
# ==============================
# 1) Import Libraries
# ==============================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
# ==============================
# 2) Read Data
# ==============================

data = pd.read_csv("ds_salaries.csv")

print(data.info())
print("\nMissing Values:\n", data.isnull().sum())

In [ ]:
# ==============================
# 3) Data Cleaning
# ==============================

data = data.drop(columns=["Unnamed: 0"], errors="ignore")
data = data.drop_duplicates()

for col in data.select_dtypes(include="object"):
    data[col] = data[col].str.strip()

data = data.drop(columns=["salary", "salary_currency"], errors="ignore")

In [ ]:
# ==============================
# 4) Feature Engineering
# ==============================

data["remote_type"] = data["remote_ratio"].map({
    0: "On-site",
    50: "Hybrid",
    100: "Remote"
})

data = data.drop(columns=["remote_ratio"])

In [ ]:
# ==============================
# 5) Remove Outliers (IQR Method)
# ==============================

Q1 = data["salary_in_usd"].quantile(0.25)
Q3 = data["salary_in_usd"].quantile(0.75)
IQR = Q3 - Q1

data = data[data["salary_in_usd"] < Q3 + 1.5 * IQR]

print(f"Salary range after removing outliers: {data['salary_in_usd'].min():,.0f} - {data['salary_in_usd'].max():,.0f}")
print(f"Rows remaining: {len(data)}")

In [ ]:
# ==============================
# 6) X and y
# ==============================

y = data["salary_in_usd"]
X = data.drop(columns=["salary_in_usd"])

In [ ]:
# ==============================
# 7) Encoding
# ==============================

categorical_cols = X.select_dtypes(include="object").columns

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ],
    remainder="passthrough"
)

In [ ]:
# ==============================
# 8) Split Data
# ==============================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# ==============================
# 9) RandomForest Model
# ==============================

model = Pipeline([
    ("prep", preprocessor),
    ("rf", RandomForestRegressor(
        n_estimators=300,
        max_depth=25,
        random_state=42
    ))
])

model.fit(X_train, y_train)

In [ ]:
# ==============================
# 10) Prediction
# ==============================

y_pred = model.predict(X_test)

In [ ]:
# ==============================
# 11) Evaluation
# ==============================

mae  = mean_absolute_error(y_test, y_pred)
mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2   = r2_score(y_test, y_pred)

print("Model Evaluation (Test Set):")
print(f"  MAE  : {mae:,.2f}")
print(f"  RMSE : {rmse:,.2f}")
print(f"  R2   : {r2:.4f}")

# Cross-Validation
cv_scores = cross_val_score(model, X, y, cv=5, scoring="r2")
print(f"\nCross-Validation R2 (5-fold): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

In [ ]:
# ==============================
# 12) Feature Importance
# ==============================

feature_names = model.named_steps["prep"].get_feature_names_out()
importances   = model.named_steps["rf"].feature_importances_

top_features = (
    pd.Series(importances, index=feature_names)
    .nlargest(10)
    .sort_values()
)

plt.figure(figsize=(8, 5))
top_features.plot(kind="barh")
plt.title("Top 10 Feature Importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

In [ ]:
# ==============================
# 13) Three Analysis Questions
# ==============================

# ----------------------------------
# Q1: Does experience level affect salary?
# ----------------------------------

salary_by_experience = data.groupby("experience_level")["salary_in_usd"].mean().sort_values()

print("Q1: Average Salary by Experience Level")
print(salary_by_experience)

plt.figure()
salary_by_experience.plot(kind="bar")
plt.title("Q1: Average Salary by Experience Level")
plt.xlabel("Experience Level")
plt.ylabel("Average Salary (USD)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


# ----------------------------------
# Q2: Do remote jobs have higher salaries?
# ----------------------------------

salary_by_remote = data.groupby("remote_type")["salary_in_usd"].mean().sort_values()

print("\nQ2: Average Salary by Remote Type")
print(salary_by_remote)

plt.figure()
salary_by_remote.plot(kind="bar")
plt.title("Q2: Average Salary by Remote Type")
plt.xlabel("Remote Type")
plt.ylabel("Average Salary (USD)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


# ----------------------------------
# Q3: Does company size affect salary?
# ----------------------------------

salary_by_company_size = data.groupby("company_size")["salary_in_usd"].mean().sort_values()

print("\nQ3: Average Salary by Company Size")
print(salary_by_company_size)

plt.figure()
salary_by_company_size.plot(kind="bar")
plt.title("Q3: Average Salary by Company Size")
plt.xlabel("Company Size")
plt.ylabel("Average Salary (USD)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()